## Experiment Logbook

This notebook is the main run log for the project. It follows the same structure as the paper:  
(1) generate candidates, (2) run unit tests to obtain verifier labels, (3) report baseline performance, (4) build pairwise ranker data, (5) train the ranker, (6) evaluate ranker-guided selection, and (7) repeat with a larger candidate set for the generalization check.

All intermediate artifacts are written under `outputs/`:
- `outputs/candidates/`: sampled programs (JSONL)
- `outputs/tests/`: unit-test outcomes (JSONL)
- `outputs/ranker_data/`: pairwise datasets + checkpoints
- `outputs/results/`: summary CSVs

Throughout, the goal is not to improve the generator, but to study selection under a limited test budget.

In [1]:
!pip -q install "transformers>=4.41.0" "accelerate>=0.30.0" "datasets>=2.19.0" "pyyaml>=6.0" "tqdm>=4.66.0" "torch>=2.1.0"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
databricks-sqlalchemy 1.0.5 requires pyarrow<17,>=14.0.1, but you have pyarrow 23.0.1 which is incompatible.
deepnote-toolkit 2.1.4 requires dill<=0.4,>=0.3.8; python_version <= "3.12", but you have dill 0.4.1 which is incompatible.
deepnote-toolkit 2.1.4 requires pyarrow<=17.0.0,>=13.0.0; python_version == "3.11" and sys_platform != "darwin", but you have pyarrow 23.0.1 which is incompatible.
deepnote-toolkit 2.1.4 requires requests<3,>=2.32.4, but you have requests 2.32.3 which is incompatible.
deepnote-toolkit 2.1.4 requires urllib3<3,>=2.6.3, but you have urllib3 2.3.0 which is incompatible.

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


## Step 1: Smoke test (small run)

Before spending time on the full benchmark, run a small smoke test (5 tasks, 4 candidates each).  
This checks that generation, unit testing, and evaluation scripts work end-to-end and that the output formats match across files.

Expected outcome: a small candidates JSONL, a matching tests JSONL, and a baseline summary without errors.

In [12]:
!python -m src.generate_candidates --config configs/exp.yaml

README.md: 6.52kB [00:00, 18.7MB/s]
openai_humaneval/test-00000-of-00001.par(…): 100%|█| 83.9k/83.9k [00:00<00:00, 3
config.json: 100%|█████████████████████████████| 663/663 [00:00<00:00, 3.62MB/s]
tokenizer_config.json: 7.30kB [00:00, 26.7MB/s]
vocab.json: 2.78MB [00:00, 110MB/s]
merges.txt: 1.67MB [00:00, 126MB/s]
tokenizer.json: 7.03MB [00:00, 148MB/s]
`torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors.index.json: 27.8kB [00:00, 72.6MB/s]
Fetching 4 files: 100%|███████████████████████████| 4/4 [01:09<00:00, 17.36s/it]
Download complete: 100%|████████████████████| 15.2G/15.2G [01:09<00:00, 219MB/s]
Loading weights: 100%|█| 339/339 [00:23<00:00, 14.23it/s, Materializing param=mo
Generating candidates: 100%|██████████████████████| 5/5 [01:50<00:00, 22.12s/it]
[OK] Wrote 20 candidates to: outputs/candidates/humaneval_candidates_20260304_145632.jsonl


In [16]:
!python -m src.run_tests \
  --config configs/exp.yaml \
  --candidates outputs/candidates/humaneval_candidates_20260304_133744.jsonl \
  --timeout_s 3.0

[OK] Wrote 20 test results to: outputs/tests/humaneval_tests_20260304_134549.jsonl


### Smoke test sanity checks

At this point, focused on consistency:
- candidate file and test file should agree on `(task_id, cand_id)`
- the number of test rows should equal `#tasks × N`
- baseline script should print reasonable pass@1 values (not necessarily high)

If these look correct, move on to the full benchmark run.

In [15]:
!python -m src.evaluate_baselines \
  --config configs/exp.yaml \
  --candidates outputs/candidates/humaneval_candidates_20260304_133744.jsonl \
  --tests outputs/tests/humaneval_tests_20260304_134549.jsonl

Tasks evaluated: 5
N candidates per task: 4
pass@1 (first-sample): 1.000
pass@1 (random, mean over 5 seeds): 1.000  (scores=[1.0, 1.0, 1.0, 1.0, 1.0])
pass@1 (test-all best-of-N): 1.000
Avg pass fraction per task (passed/N): 1.000
[OK] Wrote baseline summary to: outputs/results/baselines_20260304_150053.csv


## Step 2: Full benchmark run (HumanEval, N=8)

Now run the full HumanEval test split (164 tasks) with `N=8` candidates per task.  
This produces the main artifacts for RQ1 and RQ2, and also serves as the data source for ranker training.

This stage can take time because it includes generation and unit-test execution over the full dataset.

In [7]:
!python -m src.generate_candidates --config configs/exp.yaml

README.md: 6.52kB [00:00, 20.4MB/s]
openai_humaneval/test-00000-of-00001.par(…): 100%|█| 83.9k/83.9k [00:00<00:00, 2
config.json: 100%|█████████████████████████████| 663/663 [00:00<00:00, 4.01MB/s]
tokenizer_config.json: 7.30kB [00:00, 26.2MB/s]
vocab.json: 2.78MB [00:00, 112MB/s]
merges.txt: 1.67MB [00:00, 113MB/s]
tokenizer.json: 7.03MB [00:00, 143MB/s]
`torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors.index.json: 27.8kB [00:00, 95.7MB/s]
Fetching 4 files: 100%|███████████████████████████| 4/4 [01:10<00:00, 17.72s/it]
Download complete: 100%|████████████████████| 15.2G/15.2G [01:10<00:00, 215MB/s]
Generating candidates: 100%|████████████████| 164/164 [1:45:56<00:00, 38.76s/it]
[OK] Wrote 1312 candidates to: outputs/candidates/humaneval_candidates_20260316_132554.jsonl


In [4]:
!python -m src.run_tests \
  --config configs/exp.yaml \
  --candidates outputs/candidates/humaneval_candidates_20260316_132554.jsonl \
  --timeout_s 3.0

README.md: 6.52kB [00:00, 19.3MB/s]
openai_humaneval/test-00000-of-00001.par(…): 100%|█| 83.9k/83.9k [00:00<00:00, 1
Generating test split: 100%|████████| 164/164 [00:00<00:00, 42395.43 examples/s]
[OK] Wrote 1312 test results to: outputs/tests/humaneval_tests_20260317_143247.jsonl


In [10]:
!wc -l outputs/tests/humaneval_tests_20260317_143247.jsonl

1312 outputs/tests/humaneval_tests_20260317_143247.jsonl


In [13]:
!python -m src.evaluate_baselines \
  --config configs/exp.yaml \
  --candidates outputs/candidates/humaneval_candidates_20260316_132554.jsonl \
  --tests outputs/tests/humaneval_tests_20260317_143247.jsonl

Tasks evaluated: 164
N candidates per task: 8
pass@1 (first-sample): 0.866
pass@1 (random, mean over 5 seeds): 0.846  (scores=[0.8719512195121951, 0.8414634146341463, 0.8353658536585366, 0.8475609756097561, 0.8353658536585366])
pass@1 (test-all best-of-N): 0.915
Avg pass fraction per task (passed/N): 0.843
[OK] Wrote baseline summary to: outputs/results/baselines_20260317_143442.csv


In [16]:
!python -m src.build_ranker_dataset \
  --config configs/exp.yaml \
  --candidates outputs/candidates/humaneval_candidates_20260316_132554.jsonl \
  --tests outputs/tests/humaneval_tests_20260317_143247.jsonl \
  --pairs_per_task 30 \
  --seed 42

Tasks total: 164 | train=131 val=16 test=17
Pairs: train=570 val=120 test=120
[OK] Wrote:
  outputs/ranker_data/pairs_train_20260317_144336.jsonl
  outputs/ranker_data/pairs_val_20260317_144336.jsonl
  outputs/ranker_data/pairs_test_20260317_144336.jsonl


In [19]:
!python -m src.train_ranker \
  --config configs/exp.yaml \
  --train_path outputs/ranker_data/pairs_train_20260317_144336.jsonl \
  --val_path outputs/ranker_data/pairs_val_20260317_144336.jsonl \
  --test_path outputs/ranker_data/pairs_test_20260317_144336.jsonl \
  --encoder_name microsoft/codebert-base \
  --batch_size 16 \
  --epochs 8 \
  --max_length 384 \
  --lr 2e-5 \
  --fp16

config.json: 100%|█████████████████████████████| 498/498 [00:00<00:00, 2.95MB/s]
tokenizer_config.json: 100%|██████████████████| 25.0/25.0 [00:00<00:00, 149kB/s]
vocab.json: 899kB [00:00, 88.6MB/s]
merges.txt: 456kB [00:00, 80.2MB/s]
special_tokens_map.json: 100%|██████████████████| 150/150 [00:00<00:00, 991kB/s]
pytorch_model.bin: 100%|██████████████████████| 499M/499M [00:01<00:00, 269MB/s]
model.safetensors:   0%|                             | 0.00/499M [00:00<?, ?B/s]
Loading weights: 100%|█████████████████████| 199/199 [00:00<00:00, 23760.72it/s]
model.safetensors:   0%|                             | 0.00/499M [00:00<?, ?B/s]/datasets/_deepnote_work/src/train_ranker.py:209: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(args.fp16 and device.type == "cuda"))

Train epoch 1/8:   0%|                                   | 0/36 [00:00<?, ?it/s]/datasets/_deepnote_

### Diagnosis: overfitting / weak conditioning

The first ranker training run shows a clear pattern: training loss drops fast, validation accuracy peaks early, and test pairwise accuracy is poor.  
This suggests the model is not learning a stable notion of “correct for this task,” but instead fitting patterns that do not generalize to unseen tasks.

This is not a compute issue. It is mostly a representation issue: the ranker needs to see the task prompt together with the candidate code.

## Step 4: Fix — prompt-conditioned pair dataset

To make the supervision meaningful, I rebuild the pairwise dataset so that each example includes:
- the HumanEval prompt/specification
- the candidate code

Concretely, ranker inputs are formed as `[PROMPT] ... [CODE] ...`.  
Then I retrain with early stopping and fewer epochs to reduce overfitting.

In [7]:
!python -m src.build_ranker_dataset \
  --config configs/exp.yaml \
  --candidates outputs/candidates/humaneval_candidates_20260316_132554.jsonl \
  --tests outputs/tests/humaneval_tests_20260317_143247.jsonl \
  --pairs_per_task 120 \
  --seed 42

README.md: 6.52kB [00:00, 17.5MB/s]
openai_humaneval/test-00000-of-00001.par(…): 100%|█| 83.9k/83.9k [00:00<00:00, 2
Generating test split: 100%|████████| 164/164 [00:00<00:00, 42544.89 examples/s]
Tasks total: 164 | train=123 val=20 test=21
Pairs: train=2280 val=480 test=480
[OK] Wrote:
  outputs/ranker_data/pairs_prompt_train_20260317_153223.jsonl
  outputs/ranker_data/pairs_prompt_val_20260317_153223.jsonl
  outputs/ranker_data/pairs_prompt_test_20260317_153223.jsonl


In [13]:
!python -m src.train_ranker \
  --config configs/exp.yaml \
  --train_path outputs/ranker_data/pairs_prompt_train_20260317_153223.jsonl \
  --val_path outputs/ranker_data/pairs_prompt_val_20260317_153223.jsonl \
  --test_path outputs/ranker_data/pairs_prompt_test_20260317_153223.jsonl \
  --encoder_name microsoft/codebert-base \
  --max_length 512 \
  --batch_size 8 \
  --epochs 6 \
  --lr 2e-5 \
  --fp16

pytorch_model.bin: 100%|██████████████████████| 499M/499M [00:01<00:00, 269MB/s]
model.safetensors:   0%|                             | 0.00/499M [00:00<?, ?B/s]
Loading weights: 100%|█████████████████████| 199/199 [00:00<00:00, 26467.94it/s]
model.safetensors:  81%|█████████████████▊    | 402M/499M [00:01<00:00, 958MB/s]/datasets/_deepnote_work/src/train_ranker.py:130: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(args.fp16 and device.type == "cuda"))

Train 1/6:   0%|                                        | 0/285 [00:00<?, ?it/s]/datasets/_deepnote_work/src/train_ranker.py:141: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(args.fp16 and device.type == "cuda")):
model.safetensors: 100%|██████████████████████| 499M/499M [00:01<00:00, 348MB/s]
/d

## Step 5: Ranker-guided selection evaluation (N=8)

With the prompt-conditioned ranker trained, I evaluate selection policies on the same N=8 candidate pool.  
I report:
- ranker top-1 (selection without tests at selection time)
- ranker top-k + tests (budgeted verification)

The important comparison is whether a small test budget (e.g., k=2 or k=4) recovers most of the best-of-8 gain.

In [16]:
!python -m src.evaluate_ranker_selection \
  --config configs/exp.yaml \
  --candidates outputs/candidates/humaneval_candidates_20260316_132554.jsonl \
  --tests outputs/tests/humaneval_tests_20260317_143247.jsonl \
  --ckpt outputs/ranker_data/ranker_model_20260317_153540/best.pt \
  --encoder_name microsoft/codebert-base \
  --max_length 512 \
  --k_list 1,2,4

Loading weights: 100%|█████████████████████| 199/199 [00:00<00:00, 21043.96it/s]
Tasks evaluated: 164
N candidates per task: 8
pass@1 first-sample: 0.866
pass@1 random mean: 0.846 (scores=[0.8719512195121951, 0.8414634146341463, 0.8353658536585366, 0.8475609756097561, 0.8353658536585366])
pass@1 test-all best-of-N: 0.915
Ranker top-1: 100%|███████████████████████████| 164/164 [00:19<00:00,  8.31it/s]
pass@1 ranker top-1: 0.902
Ranker top-1+tests: 100%|█████████████████████| 164/164 [00:19<00:00,  8.26it/s]
pass@1 ranker top-1+tests: 0.902 | avg tests used per task: 1.00
Ranker top-2+tests: 100%|█████████████████████| 164/164 [00:20<00:00,  8.18it/s]
pass@1 ranker top-2+tests: 0.902 | avg tests used per task: 2.00
Ranker top-4+tests: 100%|█████████████████████| 164/164 [00:20<00:00,  8.15it/s]
pass@1 ranker top-4+tests: 0.909 | avg tests used per task: 4.00


In [19]:
!python -m src.evaluate_ranker_selection \
  --config configs/exp.yaml \
  --candidates outputs/candidates/humaneval_candidates_20260316_132554.jsonl \
  --tests outputs/tests/humaneval_tests_20260317_143247.jsonl \
  --ckpt outputs/ranker_data/ranker_model_20260317_153540/best.pt \
  --encoder_name microsoft/codebert-base \
  --max_length 512 \
  --k_list 1,2,4,8

Loading weights: 100%|█████████████████████| 199/199 [00:00<00:00, 28068.28it/s]
Tasks evaluated: 164
N candidates per task: 8
pass@1 first-sample: 0.866
pass@1 random mean: 0.846 (scores=[0.8719512195121951, 0.8414634146341463, 0.8353658536585366, 0.8475609756097561, 0.8353658536585366])
pass@1 test-all best-of-N: 0.915
Ranker top-1: 100%|███████████████████████████| 164/164 [00:19<00:00,  8.28it/s]
pass@1 ranker top-1: 0.902
Ranker top-1+tests: 100%|█████████████████████| 164/164 [00:19<00:00,  8.24it/s]
pass@1 ranker top-1+tests: 0.902 | avg tests used per task: 1.00
Ranker top-2+tests: 100%|█████████████████████| 164/164 [00:20<00:00,  8.18it/s]
pass@1 ranker top-2+tests: 0.902 | avg tests used per task: 2.00
Ranker top-4+tests: 100%|█████████████████████| 164/164 [00:20<00:00,  8.14it/s]
pass@1 ranker top-4+tests: 0.909 | avg tests used per task: 4.00
Ranker top-8+tests: 100%|█████████████████████| 164/164 [00:20<00:00,  8.11it/s]
pass@1 ranker top-8+tests: 0.915 | avg tests used 

In [22]:
import json
from collections import defaultdict

cand_path = "outputs/candidates/humaneval_candidates_20260316_132554.jsonl"
test_path = "outputs/tests/humaneval_tests_20260317_143247.jsonl"

cands = defaultdict(list)
with open(cand_path, "r", encoding="utf-8") as f:
    for line in f:
        r = json.loads(line)
        cands[r["task_id"]].append(r)
for t in cands:
    cands[t] = sorted(cands[t], key=lambda x: int(x["cand_id"]))

passed = {}
with open(test_path, "r", encoding="utf-8") as f:
    for line in f:
        r = json.loads(line)
        passed[(r["task_id"], int(r["cand_id"]))] = bool(r["passed"])

# tasks where best-of-8 passes
best_pass = []
for t, lst in cands.items():
    if any(passed.get((t, int(x["cand_id"])), False) for x in lst):
        best_pass.append(t)

print("Tasks with at least one passing candidate:", len(best_pass))

Tasks with at least one passing candidate: 150


### Error analysis (what I look for)

Error analysis separates two failure modes:
1) generation-limited tasks: no candidate passes even under best-of-N
2) selection mistakes: at least one passing candidate exists, but the ranker does not place it high enough

This helps interpret the top-1 drop and explains why increasing k (more tests) recovers accuracy.

In [25]:
!python -m src.error_analysis_shortlist \
  --candidates outputs/candidates/humaneval_candidates_20260316_132554.jsonl \
  --tests outputs/tests/humaneval_tests_20260317_143247.jsonl \
  --ckpt outputs/ranker_data/ranker_model_20260317_153540/best.pt

Loading weights: 100%|█████████████████████| 199/199 [00:00<00:00, 27188.72it/s]
No passing candidate (best-of-8 fails): 14
Best-of-8 passes but ranker top-1 fails: 2
Best-of-8 passes but ranker top-4 fails: 1

--- Missed by top-1 (task_id, first_passing_rank) ---
HumanEval/115 4
HumanEval/121 7

--- Missed by top-4 (task_id, first_passing_rank) ---
HumanEval/121 7


## Step 6: Generalization check (RQ3) — evaluate on N=16

Finally, I test whether the ranker trained under N=8 still behaves sensibly when the candidate set is larger.  
I generate a fresh candidate pool with N=16, run unit tests, and evaluate the same selection policies using the same ranker checkpoint.

This isolates generalization of the *selector* from improvements to the *generator*.

In [31]:
!python -m src.generate_candidates --config configs/exp.yaml

config.json: 100%|█████████████████████████████| 663/663 [00:00<00:00, 4.35MB/s]
tokenizer_config.json: 7.30kB [00:00, 26.4MB/s]
vocab.json: 2.78MB [00:00, 107MB/s]
merges.txt: 1.67MB [00:00, 117MB/s]
tokenizer.json: 7.03MB [00:00, 156MB/s]
`torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors.index.json: 27.8kB [00:00, 91.4MB/s]
Fetching 4 files: 100%|███████████████████████████| 4/4 [01:09<00:00, 17.28s/it]
Download complete: 100%|████████████████████| 15.2G/15.2G [01:09<00:00, 219MB/s]
Generating candidates: 100%|████████████████| 164/164 [3:28:43<00:00, 76.37s/it]
[OK] Wrote 2624 candidates to: outputs/candidates/humaneval_candidates_20260317_161503.jsonl


### Unit tests for N=16 candidates

This step runs the HumanEval unit tests for all N=16 candidates per task.  
A quick sanity check is that the test JSONL has exactly `164 × 16 = 2624` rows.

In [4]:
!python -m src.run_tests \
  --config configs/exp.yaml \
  --candidates outputs/candidates/humaneval_candidates_20260317_161503.jsonl \
  --timeout_s 3.0

README.md: 6.52kB [00:00, 19.3MB/s]
openai_humaneval/test-00000-of-00001.par(…): 100%|█| 83.9k/83.9k [00:00<00:00, 2
Generating test split: 100%|████████| 164/164 [00:00<00:00, 43729.55 examples/s]
[OK] Wrote 2624 test results to: outputs/tests/humaneval_tests_20260319_142444.jsonl


In [7]:
!wc -l outputs/tests/humaneval_tests_20260319_142444.jsonl

2624 outputs/tests/humaneval_tests_20260319_142444.jsonl


In [13]:
!python -m src.evaluate_baselines \
  --config configs/exp.yaml \
  --candidates outputs/candidates/humaneval_candidates_20260317_161503.jsonl \
  --tests outputs/tests/humaneval_tests_20260319_142444.jsonl

Tasks evaluated: 164
N candidates per task: 16
pass@1 (first-sample): 0.829
pass@1 (random, mean over 5 seeds): 0.856  (scores=[0.8536585365853658, 0.8475609756097561, 0.8475609756097561, 0.8658536585365854, 0.8658536585365854])
pass@1 (test-all best-of-N): 0.915
Avg pass fraction per task (passed/N): 0.848
[OK] Wrote baseline summary to: outputs/results/baselines_20260319_142843.csv


In [16]:
!python -m src.evaluate_ranker_selection \
  --config configs/exp.yaml \
  --candidates outputs/candidates/humaneval_candidates_20260317_161503.jsonl \
  --tests outputs/tests/humaneval_tests_20260319_142444.jsonl \
  --ckpt outputs/ranker_data/ranker_model_20260317_153540/best.pt \
  --encoder_name microsoft/codebert-base \
  --max_length 512 \
  --k_list 1,2,4,8,16

config.json: 100%|█████████████████████████████| 498/498 [00:00<00:00, 3.42MB/s]
tokenizer_config.json: 100%|██████████████████| 25.0/25.0 [00:00<00:00, 174kB/s]
vocab.json: 899kB [00:00, 98.2MB/s]
merges.txt: 456kB [00:00, 116MB/s]
special_tokens_map.json: 100%|█████████████████| 150/150 [00:00<00:00, 1.16MB/s]
pytorch_model.bin: 100%|██████████████████████| 499M/499M [00:01<00:00, 269MB/s]
Loading weights: 100%|█████████████████████| 199/199 [00:00<00:00, 16884.12it/s]

model.safetensors:   0%|                             | 0.00/499M [00:00<?, ?B/s]
model.safetensors:  13%|██▊                  | 67.1M/499M [00:00<00:01, 336MB/s]
model.safetensors:  27%|█████▉                | 134M/499M [00:00<00:01, 211MB/s]
model.safetensors:  54%|███████████▊          | 268M/499M [00:01<00:00, 370MB/s]
model.safetensors:  67%|██████████████▊       | 335M/499M [00:01<00:00, 359MB/s]
model.safetensors: 100%|██████████████████████| 499M/499M [00:01<00:00, 348MB/s]
Tasks evaluated: 164
N candidates per

In [19]:
!python -m src.error_analysis_shortlist \
  --candidates outputs/candidates/humaneval_candidates_20260317_161503.jsonl \
  --tests outputs/tests/humaneval_tests_20260319_142444.jsonl \
  --ckpt outputs/ranker_data/ranker_model_20260317_153540/best.pt

Loading weights: 100%|█████████████████████| 199/199 [00:00<00:00, 23100.48it/s]
No passing candidate (best-of-8 fails): 14
Best-of-8 passes but ranker top-1 fails: 6
Best-of-8 passes but ranker top-4 fails: 1

--- Missed by top-1 (task_id, first_passing_rank) ---
HumanEval/100 2
HumanEval/121 11
HumanEval/134 3
HumanEval/65 2
HumanEval/93 2
HumanEval/99 2

--- Missed by top-4 (task_id, first_passing_rank) ---
HumanEval/121 11


## End of run

At this point, the notebook contains all artifacts used in the paper:
- Baselines for N=8 and N=16
- Ranker training (prompt-conditioned pairs)
- Ranker-guided selection results and test-budget frontier
- Error analysis explaining remaining failures and the top-1 degradation under larger N

The remaining work is reporting: copying key numbers into tables/figures and linking the GitHub repository in the appendix.

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=280649cf-d48f-4076-bd07-3e35fb7bcf3c' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>